## Gotchas — UID/GID & the Mac/Windows sync caveat

Two problems bite almost everyone who bind-mounts, and both come from the boundary between the container and the host.

**UID / GID ownership.** A container's process runs as some numeric **UID** — whatever the image's `USER` sets, often `0` (root) or a fixed id like `999`. When that process writes to a **bind-mounted** host path, the host records the write as **that same numeric UID**, because by default Docker does *not* remap user ids (user-namespace remapping is opt-in — module 08). So a container running as root that writes `/app/out.txt` leaves a file owned by `root:root` on your host; one running as `999` leaves it owned by `999:999` — some random local user, or nobody. The standard fixes:

- **Match UIDs** — run as your host user: `docker run -u "$(id -u):$(id -g)" …`, so files come out owned by you.
- **Pre-create the host dir** with permissive ownership for dev (`mkdir -p ./data && chmod 775 ./data`). Fine for dev, not production.

**The Mac / Windows sync caveat.** On Linux the container and host share the kernel, so a bind mount is essentially free — the container sees the host inode directly. On **macOS and Windows**, Docker runs a hidden Linux VM and must bridge the host filesystem into it — and that bridge is **slow**: an `npm install` over a bind-mounted `node_modules` can be 5–10× slower than on native Linux. Mitigations, in order:

1. **Put hot directories in named volumes** (`node_modules`, `.cache/pip`, `target/`), and bind-mount only the source you actually edit. The common production-quality pattern.
2. **Use the `:cached` / `:delegated`** consistency options on binds for a speed win where strict host↔container ordering doesn't matter.

The through-line of the module: **volumes for data you keep, bind mounts for code you edit — and when a bind mount misbehaves, it's almost always UIDs or the VM bridge.**